# 02 fNIRS Pipeline Demo

This notebook demonstrates a simplified Python-based fNIRS analysis pipeline using simulated channel-level activation data.

No real subject data or sensitive information is included in this notebook.


## Analysis goals

This demo pipeline includes:

1. Simulating fNIRS channel-level activation data.
2. Defining lower-grade and upper-grade groups.
3. Comparing MA-related activation between groups for each channel.
4. Generating uncorrected p-values.
5. Applying a simple FWE correction using the Bonferroni method.
6. Visualizing significant channels.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)


## Step 1: Simulate channel-level fNIRS activation data

In the real project, this step will be replaced by processed fNIRS activation values from the lab pipeline.

Here, we create simulated data for two grade groups:

- Lower grades: G1–3
- Upper grades: G4–6


In [ ]:
n_lower = 20
n_upper = 20
n_channels = 24

channels = [f'Ch{ch:02d}' for ch in range(1, n_channels + 1)]

# Simulated lower-grade activation
lower_data = np.random.normal(loc=0.0, scale=1.0, size=(n_lower, n_channels))

# Simulated upper-grade activation
upper_data = np.random.normal(loc=0.0, scale=1.0, size=(n_upper, n_channels))

# Add artificial group differences in selected channels
effect_channels = [4, 5, 6]
upper_data[:, effect_channels] += 0.8

lower_df = pd.DataFrame(lower_data, columns=channels)
lower_df['group'] = 'G1-3'

upper_df = pd.DataFrame(upper_data, columns=channels)
upper_df['group'] = 'G4-6'

df = pd.concat([lower_df, upper_df], ignore_index=True)
df.head()


## Step 2: Run channel-wise group comparisons

For each fNIRS channel, we compare lower-grade and upper-grade activation using an independent-samples t-test.


In [ ]:
results = []

for ch in channels:
    lower_values = df.loc[df['group'] == 'G1-3', ch]
    upper_values = df.loc[df['group'] == 'G4-6', ch]
    
    t_stat, p_value = stats.ttest_ind(lower_values, upper_values, equal_var=False)
    
    results.append({
        'channel': ch,
        't_stat': t_stat,
        'p_uncorrected': p_value,
        'mean_lower': lower_values.mean(),
        'mean_upper': upper_values.mean(),
        'mean_difference_upper_minus_lower': upper_values.mean() - lower_values.mean()
    })

results_df = pd.DataFrame(results)
results_df.head()


## Step 3: Apply FWE correction

Here, we use Bonferroni correction as a simple family-wise error (FWE) correction method.


In [ ]:
alpha = 0.05
n_tests = len(channels)

results_df['p_fwe_bonferroni'] = np.minimum(results_df['p_uncorrected'] * n_tests, 1.0)
results_df['significant_uncorrected'] = results_df['p_uncorrected'] < alpha
results_df['significant_fwe'] = results_df['p_fwe_bonferroni'] < alpha

results_df.sort_values('p_uncorrected').head(10)


## Step 4: Visualize channel-wise results

This plot shows the group difference for each channel. Channels surviving FWE correction are marked in the results table.


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(results_df['channel'], results_df['mean_difference_upper_minus_lower'])
plt.axhline(0, linestyle='--', linewidth=1)
plt.xticks(rotation=90)
plt.xlabel('fNIRS channel')
plt.ylabel('Mean difference: G4-6 minus G1-3')
plt.title('Simulated MA-related activation differences by channel')
plt.tight_layout()
plt.show()


## Notes for the real analysis

In the real project, the simulated data will be replaced by processed fNIRS activation data. The expected input table may contain one row per participant and one column per channel, along with a grade-group variable.

The real analysis should carefully document:

- how activation values are computed,
- which channels are included,
- how grade groups are defined,
- which multiple-comparison correction method is used,
- and how privacy-sensitive data are excluded from GitHub.
